# Backtest Statistical Analysis

Step-by-step analysis of backtest round-trip trade results.

**What we'll cover:**
1. Load & inspect trade data
2. Draw classification (honest win rate)
3. Core metrics (win rate, EV, reward-to-risk)
4. Stress tests (monthly, quarterly, market regime breakdowns)
5. Risk metrics (max consecutive losses, max drawdown, std dev)
6. Visualizations (equity curve, P&L distribution, monthly performance)
7. Walk-forward validation (is the edge real or overfit?)

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Pretty display settings
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["figure.dpi"] = 120
plt.style.use("seaborn-v0_8-whitegrid")

# ── PROJECT PATHS ──
PROJECT_ROOT = Path(__file__).resolve().parent.parent.parent.parent if "__file__" in dir() else Path.cwd().resolve()
while not (PROJECT_ROOT / "app").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

BACKTEST_DIR = PROJECT_ROOT / "app" / "backtest"
REPORT_CSV_DIR = BACKTEST_DIR / "report" / "csv"

# ── CONFIG ──
TIMEFRAME = "15m"
INITIAL_BALANCE = 100_000

# ── LOAD DATA: Portfolio CSV (preferred) or batch CSVs (fallback) ──
PORTFOLIO_CSV = REPORT_CSV_DIR / f"backtest_trades_PORTFOLIO_{TIMEFRAME}.csv"

if PORTFOLIO_CSV.exists():
    df = pd.read_csv(PORTFOLIO_CSV)
    DATA_SOURCE = "PORTFOLIO (shared capital, chronological)"
    print(f"Loaded PORTFOLIO trades from: {PORTFOLIO_CSV.name}")
else:
    csv_files = sorted(REPORT_CSV_DIR.glob(f"backtest_trades_*_{TIMEFRAME}.csv"))
    if not csv_files:
        raise FileNotFoundError(f"No trade CSVs found in {REPORT_CSV_DIR}")
    frames = []
    for f in csv_files:
        frames.append(pd.read_csv(f))
        print(f"  {f.name}: {len(frames[-1])} trades")
    df = pd.concat(frames, ignore_index=True)
    DATA_SOURCE = f"BATCH ({len(csv_files)} symbols, independent runs)"

df["entry_time"] = pd.to_datetime(df["entry_time"])
df["exit_time"] = pd.to_datetime(df["exit_time"])
df = df.sort_values("entry_time").reset_index(drop=True)

print(f"\nData source:  {DATA_SOURCE}")
print(f"Total trades: {len(df)}")
print(f"Date range:   {df['entry_time'].min()} -> {df['exit_time'].max()}")
print(f"Symbols:      {df['symbol'].nunique()} ({', '.join(sorted(df['symbol'].unique()))})")

## Step 1: Load & Inspect Data

Load the round-trip trades CSV and get a feel for the data.

In [ ]:
print(f"Total trades: {len(df)}")
print(f"Date range:   {df['entry_time'].min()} -> {df['exit_time'].max()}")
print(f"Columns:      {list(df.columns)}")
print(f"\nTrades per symbol:")
print(df["symbol"].value_counts().to_string())
print(f"\nSide distribution:")
print(df["side"].value_counts().to_string())
df.head()

In [ ]:
# Quick summary of the P&L column
df[["pnl", "pnl_pct", "hold_duration_hours"]].describe()

## Step 1b: Draw Classification -- Honest Win Rate

### Why this matters

The strategy uses a **trailing stop-loss**: when price reaches +0.5R, the SL moves up to lock in
+0.2R of profit. If price then reverses and hits this moved SL, the trade exits with a small
positive P&L and `exit_reason="SL"`.

**The problem:** These trades are technically "wins" (pnl > 0), but they represent **zero edge** --
the strategy failed to reach its target. Counting them as wins inflates the win rate and creates
a dangerously optimistic picture.

**The fix:** Classify trades into three categories:
- **WIN**: Trade reached a take-profit target (genuine edge)
- **DRAW**: SL exit with positive P&L (trailing SL locked a tiny profit, no real edge)
- **LOSS**: Negative P&L (standard losing trade)

> **Rule of thumb:** If your "honest" win rate drops dramatically from the naive rate, your
> strategy is surviving on capital preservation, not alpha generation. That's not necessarily
> bad -- but you must know the difference.

In [ ]:
# ── DRAW CLASSIFICATION ──
# A "draw" = trailing SL locked a tiny profit but never reached TP.
# Condition: exit_reason == "SL" AND pnl > 0
df["trade_class"] = "LOSS"
df.loc[df["pnl"] > 0, "trade_class"] = "WIN"
df.loc[(df["exit_reason"] == "SL") & (df["pnl"] > 0), "trade_class"] = "DRAW"

# Count each class
class_counts = df["trade_class"].value_counts()
n_wins = class_counts.get("WIN", 0)
n_draws = class_counts.get("DRAW", 0)
n_losses = class_counts.get("LOSS", 0)
total = len(df)

print("=== TRADE CLASSIFICATION ===")
print(f"  WINS:   {n_wins:>5}  ({n_wins/total*100:5.1f}%)  -- reached TP target")
print(f"  DRAWS:  {n_draws:>5}  ({n_draws/total*100:5.1f}%)  -- trailing SL locked tiny profit")
print(f"  LOSSES: {n_losses:>5}  ({n_losses/total*100:5.1f}%)  -- standard losses")
print()

# ── NAIVE vs HONEST METRICS ──
naive_wr = (df["pnl"] > 0).sum() / total * 100
honest_wr = n_wins / total * 100

# Naive: any pnl > 0 is a "win"
naive_wins = df[df["pnl"] > 0]
naive_losses = df[df["pnl"] <= 0]
naive_avg_win = naive_wins["pnl"].mean()
naive_avg_loss = naive_losses["pnl"].mean()
naive_rr = abs(naive_avg_win / naive_avg_loss) if naive_avg_loss != 0 else float("inf")
naive_pf = naive_wins["pnl"].sum() / abs(naive_losses["pnl"].sum()) if naive_losses["pnl"].sum() != 0 else float("inf")

# Honest: only TP exits count as wins
honest_wins_df = df[df["trade_class"] == "WIN"]
honest_losses_df = df[df["trade_class"] == "LOSS"]
draws_df = df[df["trade_class"] == "DRAW"]

honest_avg_win = honest_wins_df["pnl"].mean() if len(honest_wins_df) > 0 else 0
honest_avg_loss = honest_losses_df["pnl"].mean() if len(honest_losses_df) > 0 else 0
honest_rr = abs(honest_avg_win / honest_avg_loss) if honest_avg_loss != 0 else float("inf")

# Honest profit factor: only real wins in numerator, only real losses in denominator
honest_gross_profit = honest_wins_df["pnl"].sum() if len(honest_wins_df) > 0 else 0
honest_gross_loss = abs(honest_losses_df["pnl"].sum()) if len(honest_losses_df) > 0 else 0
honest_pf = honest_gross_profit / honest_gross_loss if honest_gross_loss > 0 else float("inf")

# Break-even analysis with honest numbers
honest_breakeven_wr = 1 / (1 + honest_rr) * 100

# EV per trade doesn't change -- total P&L is the same regardless of classification
ev_per_trade = df["pnl"].mean()

print("=== NAIVE vs HONEST COMPARISON ===")
comparison = pd.DataFrame({
    "Metric": ["Win Rate", "Avg Win", "Avg Loss", "Reward:Risk", "Profit Factor", "Break-even WR"],
    "Naive (pnl>0 = win)": [
        f"{naive_wr:.1f}%", f"${naive_avg_win:,.2f}", f"${naive_avg_loss:,.2f}",
        f"{naive_rr:.2f}", f"{naive_pf:.2f}",
        f"{1/(1+naive_rr)*100:.1f}%",
    ],
    "Honest (draws excluded)": [
        f"{honest_wr:.1f}%", f"${honest_avg_win:,.2f}", f"${honest_avg_loss:,.2f}",
        f"{honest_rr:.2f}", f"{honest_pf:.2f}",
        f"{honest_breakeven_wr:.1f}%",
    ],
})
comparison.style.hide(axis="index")

In [ ]:
# ── DRAW IMPACT VISUALIZATION ──
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart: trade classification
colors_pie = ["#4CAF50", "#FF9800", "#F44336"]
ax1.pie(
    [n_wins, n_draws, n_losses],
    labels=[f"WIN ({n_wins})", f"DRAW ({n_draws})", f"LOSS ({n_losses})"],
    colors=colors_pie,
    autopct="%1.1f%%",
    startangle=90,
)
ax1.set_title("Trade Classification", fontsize=13, fontweight="bold")

# Bar chart: naive vs honest
categories = ["Win Rate (%)", "Profit Factor"]
naive_vals = [naive_wr, min(naive_pf, 5)]  # cap PF for display
honest_vals = [honest_wr, min(honest_pf, 5)]

x = np.arange(len(categories))
width = 0.3
bars1 = ax2.bar(x - width / 2, naive_vals, width, label="Naive", color="#90CAF9", edgecolor="black")
bars2 = ax2.bar(x + width / 2, honest_vals, width, label="Honest", color="#1565C0", edgecolor="black")

# Add value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax2.text(bar.get_x() + bar.get_width() / 2, h + 0.5, f"{h:.1f}",
                 ha="center", va="bottom", fontsize=10, fontweight="bold")

ax2.set_xticks(x)
ax2.set_xticklabels(categories)
ax2.set_title("Naive vs Honest Metrics", fontsize=13, fontweight="bold")
ax2.legend()
ax2.set_ylim(0, max(naive_vals + honest_vals) * 1.2)

fig.tight_layout()
plt.show()

# Print the draw P&L summary
if n_draws > 0:
    print(f"\nDraw trades summary:")
    print(f"  Average draw P&L: ${draws_df['pnl'].mean():,.2f}")
    print(f"  Total draw P&L:   ${draws_df['pnl'].sum():,.2f}")
    print(f"  This capital preservation saved you ${draws_df['pnl'].sum():,.2f} vs taking full losses.")

## Step 2: Core Metrics

The fundamental numbers every trader needs:

| Metric | Formula |
|--------|---------|
| **Win Rate** | wins / total trades |
| **Avg Win** | mean P&L of winning trades |
| **Avg Loss** | mean P&L of losing trades |
| **EV per Trade** | (win_rate × avg_win) + (loss_rate × avg_loss) |
| **Reward-to-Risk** | avg_win / \|avg_loss\| |
| **Profit Factor** | gross_profit / gross_loss |

In [ ]:
wins = df[df["pnl"] > 0]
losses = df[df["pnl"] <= 0]

total_trades = len(df)
win_count = len(wins)
loss_count = len(losses)
win_rate = win_count / total_trades * 100
loss_rate = 100 - win_rate

avg_win = wins["pnl"].mean() if win_count > 0 else 0
avg_loss = losses["pnl"].mean() if loss_count > 0 else 0

# EV = (win_rate × avg_win) + (loss_rate × avg_loss)
ev_per_trade = (win_rate / 100 * avg_win) + (loss_rate / 100 * avg_loss)

# Reward-to-Risk = avg_win / |avg_loss|
reward_to_risk = abs(avg_win / avg_loss) if avg_loss != 0 else float("inf")

# Profit Factor = gross_profit / gross_loss
gross_profit = wins["pnl"].sum() if win_count > 0 else 0
gross_loss = abs(losses["pnl"].sum()) if loss_count > 0 else 0
profit_factor = gross_profit / gross_loss if gross_loss > 0 else float("inf")

total_pnl = df["pnl"].sum()

# Display as a clean table
core_metrics = pd.DataFrame({
    "Metric": [
        "Total Trades", "Winning Trades", "Losing Trades",
        "Win Rate (%)", "Avg Win ($)", "Avg Loss ($)",
        "EV per Trade ($)", "Reward-to-Risk", "Profit Factor",
        "Total P&L ($)",
    ],
    "Value": [
        f"{total_trades}", f"{win_count}", f"{loss_count}",
        f"{win_rate:.2f}%", f"${avg_win:,.2f}", f"${avg_loss:,.2f}",
        f"${ev_per_trade:,.2f}", f"{reward_to_risk:.2f}", f"{profit_factor:.2f}",
        f"${total_pnl:,.2f}",
    ],
})
core_metrics.style.hide(axis="index")

### How to interpret

- **Win rate > 50%** is good, but only matters in combination with reward-to-risk
- **EV per trade > 0** means the strategy is profitable on average
- **Reward-to-Risk > 1** means wins are bigger than losses (you can be profitable even with <50% win rate)
- **Profit Factor > 1.5** is considered a solid edge

**The key relationship:**
- If R:R is low (e.g. 0.5), you need a very high win rate (~67%+) to break even
- If R:R is high (e.g. 2.0), you only need ~33% win rate to break even
- Break-even win rate = 1 / (1 + R:R)

In [ ]:
# Break-even analysis
breakeven_wr = 1 / (1 + reward_to_risk) * 100
print(f"Your R:R = {reward_to_risk:.2f}")
print(f"Break-even win rate needed = {breakeven_wr:.1f}%")
print(f"Your actual win rate       = {win_rate:.1f}%")
print(f"Gap                        = {win_rate - breakeven_wr:+.1f}% {'(EDGE EXISTS)' if win_rate > breakeven_wr else '(NO EDGE - need higher R:R or win rate)'}")

## Step 3: Stress Tests

Break down performance by time period and market condition to see if the edge is consistent or driven by a few lucky months.

### 3a. Monthly Breakdown

In [ ]:
df["_month"] = df["exit_time"].dt.to_period("M")

monthly = df.groupby("_month").agg(
    trades=("pnl", "count"),
    wins=("pnl", lambda x: (x > 0).sum()),
    pnl=("pnl", "sum"),
    avg_pnl=("pnl", "mean"),
).reset_index()

monthly["win_rate"] = (monthly["wins"] / monthly["trades"] * 100).round(1)
monthly["month"] = monthly["_month"].astype(str)
monthly[["month", "trades", "wins", "win_rate", "pnl", "avg_pnl"]]

### 3b. Quarterly Breakdown

In [ ]:
df["_quarter"] = df["exit_time"].dt.to_period("Q")

quarterly = df.groupby("_quarter").agg(
    trades=("pnl", "count"),
    wins=("pnl", lambda x: (x > 0).sum()),
    pnl=("pnl", "sum"),
).reset_index()

quarterly["win_rate"] = (quarterly["wins"] / quarterly["trades"] * 100).round(1)
quarterly["quarter"] = quarterly["_quarter"].astype(str)
quarterly[["quarter", "trades", "wins", "win_rate", "pnl"]]

### 3c. Market Regime Breakdown

Classify each trade into a market regime based on price movement between consecutive trades:
- **TRENDING_UP**: entry price rose >1% vs previous trade's entry
- **TRENDING_DOWN**: entry price fell >1%
- **RANGING**: within +/-1%

This helps answer: *"Does the strategy work better in trends or ranges?"*

In [ ]:
tmp = df.sort_values("entry_time").reset_index(drop=True)
prev_price = tmp["entry_price"].shift(1)
pct_change = (tmp["entry_price"] - prev_price) / prev_price * 100

conditions = [pct_change > 1.0, pct_change < -1.0]
choices = ["TRENDING_UP", "TRENDING_DOWN"]
tmp["regime"] = np.select(conditions, choices, default="RANGING")
tmp.loc[0, "regime"] = "RANGING"  # first trade has no prior

regime = tmp.groupby("regime").agg(
    trades=("pnl", "count"),
    wins=("pnl", lambda x: (x > 0).sum()),
    pnl=("pnl", "sum"),
    avg_pnl=("pnl", "mean"),
).reset_index()

regime["win_rate"] = (regime["wins"] / regime["trades"] * 100).round(1)
regime

### 3d. Exit Reason Breakdown

How are trades exiting? This tells you if your TP/SL levels are well-calibrated.

In [ ]:
exit_stats = df.groupby("exit_reason").agg(
    trades=("pnl", "count"),
    wins=("trade_class", lambda x: (x == "WIN").sum()),
    draws=("trade_class", lambda x: (x == "DRAW").sum()),
    losses=("trade_class", lambda x: (x == "LOSS").sum()),
    avg_pnl=("pnl", "mean"),
    total_pnl=("pnl", "sum"),
).sort_values("trades", ascending=False)

exit_stats

## Step 4: Risk Metrics

These metrics help you understand the *worst-case* scenarios and overall risk profile.

In [ ]:
# ── Max consecutive wins/losses ──
def max_consecutive(series, value):
    max_count = current = 0
    for v in series:
        if v == value:
            current += 1
            max_count = max(max_count, current)
        else:
            current = 0
    return max_count

is_win = (df["pnl"] > 0).astype(int)
max_consec_wins = max_consecutive(is_win, 1)
max_consec_losses = max_consecutive(is_win, 0)

# ── Equity curve & drawdown ──
equity = [INITIAL_BALANCE]
for pnl in df["pnl"]:
    equity.append(equity[-1] + pnl)

peak = equity[0]
drawdowns = []
max_dd = max_dd_val = 0.0
for val in equity:
    if val > peak:
        peak = val
    dd = (peak - val) / peak if peak > 0 else 0
    drawdowns.append(dd * 100)
    if dd > max_dd:
        max_dd = dd
        max_dd_val = peak - val

# ── Return distribution stats ──
returns_pct = df["pnl_pct"].values
std_dev = np.std(returns_pct, ddof=1) if len(returns_pct) > 1 else 0
mean_ret = np.mean(returns_pct)
sharpe = mean_ret / std_dev if std_dev > 0 else 0

neg_returns = returns_pct[returns_pct < 0]
downside_std = np.std(neg_returns, ddof=1) if len(neg_returns) > 1 else 0
sortino = mean_ret / downside_std if downside_std > 0 else 0

var_95 = np.percentile(returns_pct, 5) if len(returns_pct) >= 5 else min(returns_pct)

# Display
risk_metrics = pd.DataFrame({
    "Metric": [
        "Max Consecutive Wins", "Max Consecutive Losses",
        "Max Drawdown ($)", "Max Drawdown (%)",
        "Std Dev of Returns (%)", "Sharpe Ratio (trade-level)",
        "Sortino Ratio (trade-level)", "Value-at-Risk 95% (%)",
    ],
    "Value": [
        f"{max_consec_wins}", f"{max_consec_losses}",
        f"${max_dd_val:,.2f}", f"{max_dd * 100:.2f}%",
        f"{std_dev:.2f}%", f"{sharpe:.3f}",
        f"{sortino:.3f}", f"{var_95:.2f}%",
    ],
})
risk_metrics.style.hide(axis="index")

### How to interpret risk metrics

- **Max Consecutive Losses**: How many losses in a row you should mentally prepare for. If this is 5+, you need strong discipline.
- **Max Drawdown**: The worst peak-to-trough decline. A 20%+ drawdown is psychologically tough to endure live.
- **Sharpe Ratio**: Return per unit of risk. Above 1.0 is good, above 2.0 is excellent.
- **Sortino Ratio**: Like Sharpe but only penalizes downside volatility. Better metric for asymmetric strategies.
- **VaR 95%**: "In the worst 5% of trades, you'll lose at least X%". This is your tail risk.

## Step 5: Visualizations

### 5a. Equity Curve

The most important chart — shows how your balance evolved over time.

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

# Equity curve
ax1.plot(range(len(equity)), equity, linewidth=1.5, color="#2196F3")
ax1.axhline(y=INITIAL_BALANCE, color="gray", linestyle="--", alpha=0.5, label="Initial Balance")
ax1.fill_between(range(len(equity)), equity, INITIAL_BALANCE,
                  where=[e >= INITIAL_BALANCE for e in equity], alpha=0.15, color="green")
ax1.fill_between(range(len(equity)), equity, INITIAL_BALANCE,
                  where=[e < INITIAL_BALANCE for e in equity], alpha=0.15, color="red")
ax1.set_title("Equity Curve", fontsize=14, fontweight="bold")
ax1.set_ylabel("Balance ($)")
ax1.legend()

# Drawdown subplot
ax2.fill_between(range(len(drawdowns)), drawdowns, color="red", alpha=0.3)
ax2.plot(range(len(drawdowns)), drawdowns, color="red", linewidth=0.8)
ax2.set_title("Drawdown (%)", fontsize=11)
ax2.set_xlabel("Trade #")
ax2.set_ylabel("DD %")
ax2.invert_yaxis()  # drawdown goes down

fig.tight_layout()
plt.show()

### 5b. Win/Loss Distribution

Histogram showing the spread of trade outcomes. Ideally you want the green (wins) clustered to the right and losses tightly clustered near zero.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# P&L in dollars
bin_edges = np.linspace(df["pnl"].min(), df["pnl"].max(), 25)
ax1.hist(wins["pnl"], bins=bin_edges, color="#4CAF50", alpha=0.7, label=f"Wins ({len(wins)})")
ax1.hist(losses["pnl"], bins=bin_edges, color="#F44336", alpha=0.7, label=f"Losses ({len(losses)})")
ax1.axvline(x=0, color="black", linewidth=0.8)
ax1.axvline(x=df["pnl"].mean(), color="blue", linestyle="--", label=f"Mean: ${df['pnl'].mean():,.0f}")
ax1.set_title("P&L Distribution ($)", fontsize=13, fontweight="bold")
ax1.set_xlabel("P&L ($)")
ax1.set_ylabel("Frequency")
ax1.legend()

# P&L in percent
bin_edges_pct = np.linspace(df["pnl_pct"].min(), df["pnl_pct"].max(), 25)
ax2.hist(wins["pnl_pct"], bins=bin_edges_pct, color="#4CAF50", alpha=0.7, label="Wins")
ax2.hist(losses["pnl_pct"], bins=bin_edges_pct, color="#F44336", alpha=0.7, label="Losses")
ax2.axvline(x=0, color="black", linewidth=0.8)
ax2.axvline(x=df["pnl_pct"].mean(), color="blue", linestyle="--", label=f"Mean: {df['pnl_pct'].mean():.1f}%")
ax2.set_title("P&L Distribution (%)", fontsize=13, fontweight="bold")
ax2.set_xlabel("P&L (%)")
ax2.set_ylabel("Frequency")
ax2.legend()

fig.tight_layout()
plt.show()

### 5c. Monthly Performance

Bar chart of P&L per month with win rate overlay. Helps spot seasonality.

In [ ]:
fig, ax1 = plt.subplots(figsize=(12, 6))

colors = ["#4CAF50" if p >= 0 else "#F44336" for p in monthly["pnl"]]
x = range(len(monthly))
ax1.bar(x, monthly["pnl"], color=colors, alpha=0.7, label="P&L ($)")
ax1.set_ylabel("P&L ($)")
ax1.axhline(y=0, color="gray", linestyle="-", linewidth=0.5)

# Win rate on secondary axis
ax2 = ax1.twinx()
ax2.plot(list(x), monthly["win_rate"].values, "o-", color="#FF9800", linewidth=2, markersize=8, label="Win Rate %")
ax2.set_ylabel("Win Rate (%)", color="#FF9800")
ax2.set_ylim(0, 100)

ax1.set_xticks(list(x))
ax1.set_xticklabels(monthly["month"], rotation=45, ha="right")
ax1.set_title("Monthly Performance", fontsize=14, fontweight="bold")

# Combined legend
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")

fig.tight_layout()
plt.show()

### 5d. Trade-by-Trade P&L (Waterfall)

Each bar is one trade — green for wins, red for losses. Helps you visually spot streaks.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

colors = ["#4CAF50" if p > 0 else "#F44336" for p in df["pnl"]]
ax.bar(range(len(df)), df["pnl"], color=colors, alpha=0.8, width=0.8)
ax.axhline(y=0, color="black", linewidth=0.5)
ax.set_title("Trade-by-Trade P&L", fontsize=14, fontweight="bold")
ax.set_xlabel("Trade #")
ax.set_ylabel("P&L ($)")

# Annotate the biggest win and biggest loss
best_idx = df["pnl"].idxmax()
worst_idx = df["pnl"].idxmin()
ax.annotate(f"Best: ${df.loc[best_idx, 'pnl']:,.0f}", xy=(best_idx, df.loc[best_idx, "pnl"]),
            fontsize=9, ha="center", va="bottom", color="green", fontweight="bold")
ax.annotate(f"Worst: ${df.loc[worst_idx, 'pnl']:,.0f}", xy=(worst_idx, df.loc[worst_idx, "pnl"]),
            fontsize=9, ha="center", va="top", color="red", fontweight="bold")

fig.tight_layout()
plt.show()

## Summary & Next Steps

Run this cell to get a quick verdict on your strategy.

In [ ]:
print("=" * 60)
print("  STRATEGY VERDICT")
print("=" * 60)

checks = {
    "EV per trade > 0": ev_per_trade > 0,
    "Profit factor > 1.0": profit_factor > 1.0,
    "Reward-to-Risk > 1.0": reward_to_risk > 1.0,
    "Win rate > break-even": win_rate > breakeven_wr,
    "Max drawdown < 20%": (max_dd * 100) < 20,
    "Sharpe > 0.5": sharpe > 0.5,
    "Max consec losses <= 5": max_consec_losses <= 5,
    "Honest win rate > break-even": honest_wr > honest_breakeven_wr,
    "Draw ratio < 30%": (n_draws / total * 100) < 30,
}

for check, passed in checks.items():
    icon = "PASS" if passed else "FAIL"
    print(f"  [{icon}] {check}")

passed_count = sum(checks.values())
total_checks = len(checks)
print(f"\n  Score: {passed_count}/{total_checks}")

if passed_count >= 7:
    print("  >> Strategy shows promise. Consider forward-testing.")
elif passed_count >= 4:
    print("  >> Mixed results. Needs optimization before live trading.")
else:
    print("  >> Strategy needs significant work. Focus on improving R:R or reducing losses.")

## Step 7: Walk-Forward Validation -- Is the Edge Real?

### The overfitting problem

A backtest always looks at historical data. The metrics above tell you how the strategy
*would have* performed -- but not how it *will* perform going forward. The danger is
**overfitting**: the strategy's parameters may have been (consciously or unconsciously) tuned
to fit the specific patterns in this historical data.

**Think of it like a student studying for an exam:**
- **In-Sample (IS)** = the practice problems you studied
- **Out-of-Sample (OOS)** = the actual exam questions you've never seen

If you only score well on practice problems but bomb the exam, you memorized answers
instead of learning the material. Same idea with trading strategies.

### Anchored walk-forward

We split the data chronologically with an **expanding window**:

```
Fold 1:  [==IS==][OOS]
Fold 2:  [===IS===][OOS]
Fold 3:  [====IS====][OOS]
Fold 4:  [=====IS=====][OOS]
```

Each fold adds one more month to the IS window and tests on the next unseen month.
This simulates how a trader would evaluate the strategy over time -- you always have
more history behind you as time goes on.

### What to look for
- **OOS metrics close to IS** -> edge is real, strategy generalizes
- **OOS much worse than IS** -> possible overfitting
- **OOS highly variable** -> strategy may be regime-dependent
- **OOS declining over time** -> edge may be decaying

In [ ]:
# ── WALK-FORWARD VALIDATION ──
MIN_IS_MONTHS = 2  # minimum months in the in-sample window
MIN_TRADES_PER_FOLD = 5  # skip OOS folds with too few trades

df["_exit_month"] = df["exit_time"].dt.to_period("M")
months = sorted(df["_exit_month"].unique())

print(f"Total months available: {len(months)} ({months[0]} to {months[-1]})")
print(f"IS starts with {MIN_IS_MONTHS} months, then expands by 1 each fold")
print()


def _fold_metrics(sub_df):
    """Compute core metrics for a walk-forward fold."""
    n = len(sub_df)
    if n == 0:
        return {"trades": 0, "win_rate": 0, "honest_wr": 0, "ev": 0, "pf": 0, "total_pnl": 0}

    w_naive = (sub_df["pnl"] > 0).sum()
    w_honest = (sub_df["trade_class"] == "WIN").sum() if "trade_class" in sub_df.columns else w_naive

    gross_p = sub_df[sub_df["pnl"] > 0]["pnl"].sum()
    gross_l = abs(sub_df[sub_df["pnl"] <= 0]["pnl"].sum())

    return {
        "trades": n,
        "win_rate": round(w_naive / n * 100, 1),
        "honest_wr": round(w_honest / n * 100, 1),
        "ev": round(float(sub_df["pnl"].mean()), 2),
        "pf": round(gross_p / gross_l, 2) if gross_l > 0 else 99.0,
        "total_pnl": round(float(sub_df["pnl"].sum()), 2),
    }


wf_results = []
for i in range(MIN_IS_MONTHS, len(months)):
    is_months = months[:i]
    oos_month = months[i]

    is_df = df[df["_exit_month"].isin(is_months)]
    oos_df = df[df["_exit_month"] == oos_month]

    if len(oos_df) < MIN_TRADES_PER_FOLD:
        continue

    is_m = _fold_metrics(is_df)
    oos_m = _fold_metrics(oos_df)

    wf_results.append({
        "fold": len(wf_results) + 1,
        "is_period": f"{is_months[0]}..{is_months[-1]}",
        "oos_period": str(oos_month),
        "is_trades": is_m["trades"],
        "oos_trades": oos_m["trades"],
        "is_wr": is_m["honest_wr"],
        "oos_wr": oos_m["honest_wr"],
        "is_ev": is_m["ev"],
        "oos_ev": oos_m["ev"],
        "is_pf": is_m["pf"],
        "oos_pf": oos_m["pf"],
        "oos_pnl": oos_m["total_pnl"],
    })

wf_df = pd.DataFrame(wf_results)
print(f"Walk-forward folds: {len(wf_df)}")
print()
wf_df

In [ ]:
# ── WALK-FORWARD CHARTS ──
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

folds = wf_df["fold"]
oos_labels = wf_df["oos_period"].tolist()

# 1. IS vs OOS Win Rate (honest)
ax = axes[0, 0]
ax.plot(folds, wf_df["is_wr"], "o-", color="#2196F3", label="IS Win Rate", linewidth=2)
ax.plot(folds, wf_df["oos_wr"], "s-", color="#F44336", label="OOS Win Rate", linewidth=2)
ax.fill_between(folds, wf_df["is_wr"], wf_df["oos_wr"], alpha=0.1, color="red")
ax.set_title("Honest Win Rate: IS vs OOS", fontweight="bold")
ax.set_xlabel("Fold")
ax.set_ylabel("Win Rate (%)")
ax.set_xticks(folds)
ax.set_xticklabels(oos_labels, rotation=45, ha="right", fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3)

# 2. IS vs OOS EV
ax = axes[0, 1]
ax.plot(folds, wf_df["is_ev"], "o-", color="#2196F3", label="IS EV/Trade", linewidth=2)
ax.plot(folds, wf_df["oos_ev"], "s-", color="#F44336", label="OOS EV/Trade", linewidth=2)
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("EV per Trade: IS vs OOS ($)", fontweight="bold")
ax.set_xlabel("Fold")
ax.set_ylabel("EV ($)")
ax.set_xticks(folds)
ax.set_xticklabels(oos_labels, rotation=45, ha="right", fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3)

# 3. IS vs OOS Profit Factor
ax = axes[1, 0]
pf_cap = 3.0
ax.plot(folds, wf_df["is_pf"].clip(upper=pf_cap), "o-", color="#2196F3", label="IS PF", linewidth=2)
ax.plot(folds, wf_df["oos_pf"].clip(upper=pf_cap), "s-", color="#F44336", label="OOS PF", linewidth=2)
ax.axhline(y=1.0, color="gray", linestyle="--", alpha=0.5, label="Break-even (PF=1)")
ax.set_title("Profit Factor: IS vs OOS", fontweight="bold")
ax.set_xlabel("Fold")
ax.set_ylabel("Profit Factor")
ax.set_xticks(folds)
ax.set_xticklabels(oos_labels, rotation=45, ha="right", fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3)

# 4. Cumulative OOS P&L
ax = axes[1, 1]
cum_oos_pnl = wf_df["oos_pnl"].cumsum()
colors_bar = ["#4CAF50" if p >= 0 else "#F44336" for p in wf_df["oos_pnl"]]
ax.bar(folds, wf_df["oos_pnl"], color=colors_bar, alpha=0.7, label="OOS P&L per fold")
ax.plot(folds, cum_oos_pnl, "k-", linewidth=2, marker="o", markersize=5, label="Cumulative OOS P&L")
ax.axhline(y=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Out-of-Sample P&L by Fold", fontweight="bold")
ax.set_xlabel("Fold")
ax.set_ylabel("P&L ($)")
ax.set_xticks(folds)
ax.set_xticklabels(oos_labels, rotation=45, ha="right", fontsize=8)
ax.legend()
ax.grid(True, alpha=0.3)

fig.suptitle("Walk-Forward Validation Results", fontsize=15, fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

In [ ]:
# ── WALK-FORWARD ROBUSTNESS VERDICT ──

def _max_consec_neg(series):
    """Count longest streak of non-positive values."""
    mx = cur = 0
    for v in series:
        if v <= 0:
            cur += 1
            mx = max(mx, cur)
        else:
            cur = 0
    return mx


median_oos_pf = wf_df["oos_pf"].median()
median_oos_wr = wf_df["oos_wr"].median()
median_oos_ev = wf_df["oos_ev"].median()

oos_profitable_folds = (wf_df["oos_pnl"] > 0).sum()
oos_profitable_pct = oos_profitable_folds / len(wf_df) * 100

# IS-OOS gaps: how much does performance degrade out-of-sample?
wr_gap = wf_df["is_wr"].median() - median_oos_wr
ev_gap = wf_df["is_ev"].median() - median_oos_ev
max_consec_oos_loss = _max_consec_neg(wf_df["oos_pnl"])

print("=" * 60)
print("  WALK-FORWARD ROBUSTNESS VERDICT")
print("=" * 60)
print(f"\n  Folds analyzed:            {len(wf_df)}")
print(f"  Median OOS Win Rate:       {median_oos_wr:.1f}%  (honest)")
print(f"  Median OOS Profit Factor:  {median_oos_pf:.2f}")
print(f"  Median OOS EV/Trade:       ${median_oos_ev:,.2f}")
print(f"  OOS Profitable Folds:      {oos_profitable_folds}/{len(wf_df)} ({oos_profitable_pct:.0f}%)")
print(f"\n  IS->OOS Win Rate Gap:      {wr_gap:+.1f} pp")
print(f"  IS->OOS EV Gap:            ${ev_gap:+,.2f}")
print(f"  Max Consec OOS Loss Folds: {max_consec_oos_loss}")

# Verdict logic
if median_oos_pf > 1.0 and oos_profitable_pct > 60:
    verdict = "ROBUST"
    verdict_msg = "Edge persists out-of-sample. Strategy generalizes well."
elif median_oos_pf > 0.8 or oos_profitable_pct > 50:
    verdict = "MARGINAL"
    verdict_msg = "Some evidence of edge, but inconsistent. Proceed with caution."
else:
    verdict = "OVERFIT"
    verdict_msg = "Edge does not survive out-of-sample testing. Likely overfit to historical data."

print(f"\n  >> VERDICT: {verdict}")
print(f"  >> {verdict_msg}")

# Detailed checks
print("\n  Detailed checks:")
checks_wf = {
    "Median OOS PF > 1.0": median_oos_pf > 1.0,
    ">60% OOS folds profitable": oos_profitable_pct > 60,
    "IS-OOS WR gap < 10pp": abs(wr_gap) < 10,
    "Median OOS EV > 0": median_oos_ev > 0,
    "Max consec OOS losing folds <= 2": max_consec_oos_loss <= 2,
}
for check, passed in checks_wf.items():
    icon = "PASS" if passed else "FAIL"
    print(f"  [{icon}] {check}")

### How to interpret walk-forward results

**Reading the charts:**
- If the red line (OOS) tracks the blue line (IS), the strategy generalizes well
- A persistent gap where OOS underperforms IS indicates some degree of overfitting
- If OOS metrics bounce wildly, the strategy may be regime-dependent (works in trends but not ranges, or vice versa)

**The IS-OOS gap:**
- **< 5 percentage points** in win rate: Excellent stability
- **5-15 pp**: Moderate degradation, acceptable if still profitable
- **> 15 pp**: Significant overfitting concern

**Cumulative OOS P&L:**
- Should trend upward over time for a robust strategy
- Flat or declining suggests the edge is not real
- A sharp drop followed by recovery may indicate regime sensitivity

**Next steps based on verdict:**
- **ROBUST**: Consider paper trading to validate with live market microstructure
- **MARGINAL**: Investigate which regimes work/fail; consider if the edge is worth the risk
- **OVERFIT**: Do NOT trade live. Re-examine strategy parameters for curve-fitting